In [ ]:
import os    

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  BERT-G3CN  |  SMS Spam Detection                                ║
# ║  Cell 1 — Environment Setup                                      ║
# ╚══════════════════════════════════════════════════════════════════╝

# ── Kaggle: install missing packages ──────────────────────────────
# !pip install -q transformers==4.40.0 spacy
# !python -m spacy download en_core_web_sm -q

import os, gc, json, random, time, warnings
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 
import seaborn as sns 
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader     
from torch.optim import AdamW 

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    roc_auc_score, roc_curve, confusion_matrix, classification_report 
)

import transformers
from transformers import BertTokenizer, BertModel, get_linear_schedule_with_warmup  
import spacy

# ── Silence noisy library output ──────────────────────────────────
warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false" 
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
transformers.logging.set_verbosity_error()

# ── Reproducibility ───────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED) 
torch.manual_seed(SEED)  
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")   

# ── spaCy ─────────────────────────────────────────────────────────
try:
    NLP      = spacy.load("en_core_web_sm", disable=["ner"])
    NLP_FULL = spacy.load("en_core_web_sm")
except OSError:
    os.system("python -m spacy download en_core_web_sm -q")
    NLP      = spacy.load("en_core_web_sm", disable=["ner"])
    NLP_FULL = spacy.load("en_core_web_sm")

# ── Confirmation ──────────────────────────────────────────────────
print("Setup complete.")
print(f"  Device : {DEVICE}" + (f"  ({torch.cuda.get_device_name(0)})" if DEVICE.type == "cuda" else ""))
print(f"  PyTorch  {torch.__version__}  |  Transformers  {transformers.__version__}")

Setup complete.
  Device : cuda  (Tesla T4)
  PyTorch  2.10.0+cu128  |  Transformers  5.0.0


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  Cell 2 — Core Definitions                                      ║
# ║  Vocabulary, graphs, model, dataset, train/eval helpers.        ║
# ║  Must run before any experiment cell.                           ║
# ╚══════════════════════════════════════════════════════════════════╝


# ══════════════════════════════════════════════════════════════════
# 2.1  Data helpers
# ══════════════════════════════════════════════════════════════════

def load_uci(path):
    df = pd.read_csv(path, sep="\t", header=None, names=["label_str", "message"])
    df["label"] = (df["label_str"].str.strip().str.lower() == "spam").astype(int)
    return df[["message", "label"]].dropna().reset_index(drop=True)


def load_exais(path):
    df = pd.read_csv(path)
    df = df.rename(columns={"Message": "message", "Spam": "label"}) 
    df["label"] = df["label"].astype(int)
    return df[["message", "label"]].dropna().reset_index(drop=True)


def split_data(texts, labels, test_size, val_frac, seed=SEED):
    tr_va, te, tr_va_l, te_l = train_test_split(
        texts, labels, test_size=test_size,
        random_state=seed, stratify=labels) 
    tr, va, tr_l, va_l = train_test_split(
        tr_va, tr_va_l,
        test_size=round(val_frac / (1.0 - test_size), 4), 
        random_state=seed, stratify=tr_va_l)
    return tr, va, te, tr_l, va_l, te_l


# ══════════════════════════════════════════════════════════════════
# 2.2  Vocabulary
# ══════════════════════════════════════════════════════════════════

def build_vocabulary(texts, min_freq=2):
    counts = Counter(
        tok.text.lower()
        for txt in texts
        for tok in NLP(str(txt))
        if not tok.is_punct and not tok.is_space
    )
    vocab, idx = {}, 0
    for w, c in counts.items():
        if c >= min_freq:
            vocab[w] = idx
            idx += 1
    return vocab


# ══════════════════════════════════════════════════════════════════
# 2.3  Graph builders  (paper §3.2)
# ══════════════════════════════════════════════════════════════════

def _sym_norm(A):
    deg        = A.sum(1) + 1e-9
    d_inv_sqrt = 1.0 / np.sqrt(deg)
    return ((A * d_inv_sqrt[:, None]) * d_inv_sqrt[None, :]).astype(np.float32)


def build_cooccurrence(texts, vocab, window=5):
    V  = len(vocab)
    co = defaultdict(float)
    for txt in tqdm(texts, desc="  co-occurrence", leave=False):
        toks = [t.text.lower() for t in NLP(str(txt))
                if not t.is_punct and not t.is_space
                and t.text.lower() in vocab]
        for i, u in enumerate(toks):
            ui = vocab[u]
            if ui >= V: continue
            for j in range(i + 1, min(i + window + 1, len(toks))):
                vi = vocab.get(toks[j], -1)
                if vi < 0 or vi >= V or u == toks[j]: continue
                key = (min(ui, vi), max(ui, vi))
                co[key] += 1.0
    A = np.zeros((V, V), dtype=np.float32)
    for (i, j), w in co.items():
        if i < V and j < V:
            A[i, j] = w; A[j, i] = w
    return _sym_norm(A)


def build_heterogeneous(texts, vocab, sim_threshold=0.70):
    V        = len(vocab)
    pos_cnts = defaultdict(float)
    ner_cnts = defaultdict(float)
    pos_tags, ner_tags = set(), set()

    for txt in tqdm(texts, desc="  heterogeneous", leave=False):
        doc = NLP_FULL(str(txt))
        for tok in doc:
            wi = vocab.get(tok.text.lower(), -1)
            if 0 <= wi < V:
                pos_cnts[(tok.text.lower(), tok.pos_)] += 1.0
                pos_tags.add(tok.pos_)
        for ent in doc.ents:
            for tok in ent:
                wi = vocab.get(tok.text.lower(), -1)
                if 0 <= wi < V:
                    ner_cnts[(tok.text.lower(), ent.label_)] += 1.0
                    ner_tags.add(ent.label_)

    def to_adj(cnts, tags):
        tags = sorted(tags) if tags else ["_"]
        mat  = np.zeros((V, len(tags)), dtype=np.float32)
        tot  = sum(cnts.values()) + 1e-9
        for (w, t), c in cnts.items():
            wi = vocab.get(w, -1)
            if 0 <= wi < V and t in tags:
                mat[wi, tags.index(t)] = c / tot
        n = np.linalg.norm(mat, axis=1, keepdims=True) + 1e-9
        return (mat / n) @ (mat / n).T

    A_wp = to_adj(pos_cnts, pos_tags)
    A_wn = to_adj(ner_cnts, ner_tags)
    np.fill_diagonal(A_wp, 0); np.fill_diagonal(A_wn, 0)

    words = list(vocab.keys())
    DIM   = 128
    def cvec(w):
        v = np.zeros(DIM, dtype=np.float32)
        for k in range(max(1, len(w) - 2)):
            v[hash(w[k:k+3]) % DIM] += 1.0
        return v / (np.linalg.norm(v) + 1e-9)
    cvecs = np.stack([cvec(w) for w in words])
    A_sem = np.zeros((V, V), dtype=np.float32)
    for s in range(0, V, 256):
        e = min(s + 256, V)
        ri, ci = np.where(cvecs[s:e] @ cvecs.T > sim_threshold)
        for r, c in zip(ri, ci):
            gr = s + r
            if gr != c:
                A_sem[gr, c] = float((cvecs[s:e] @ cvecs.T)[r, c])
    np.fill_diagonal(A_sem, 0)

    A = (A_wp + A_wn + A_sem) / 3.0
    A = (A + A.T) / 2.0
    np.fill_diagonal(A, 0)
    return _sym_norm(A)


def build_syntactic(texts, vocab, delta=1e-4):
    V    = len(vocab)
    Csyn = defaultdict(float)
    for txt in tqdm(texts, desc="  syntactic    ", leave=False):
        for tok in NLP_FULL(str(txt)):
            h = tok.head.text.lower(); d = tok.text.lower()
            hi = vocab.get(h, -1);    di = vocab.get(d, -1)
            if 0 <= hi < V and 0 <= di < V and h != d:
                Csyn[(hi, di)] += 1.0
    A = np.zeros((V, V), dtype=np.float32)
    for (i, j), c in Csyn.items():
        if i < V and j < V:
            sym = c + Csyn.get((j, i), 0.0)
            A[i, j] = sym; A[j, i] = sym
    total = A.sum() + 1e-9
    A /= total
    A[A < delta] = 0.0
    return _sym_norm(A)


def build_all_graphs(texts, vocab):
    A_co  = build_cooccurrence(texts,  vocab)
    A_het = build_heterogeneous(texts, vocab)
    A_syn = build_syntactic(texts,     vocab)
    to_t  = lambda A: torch.tensor(A, dtype=torch.float32).to(DEVICE)
    return to_t(A_co), to_t(A_het), to_t(A_syn)


# ══════════════════════════════════════════════════════════════════
# 2.4  Vocabulary → BERT embeddings
# ══════════════════════════════════════════════════════════════════

@torch.no_grad()
def build_vocab_embeddings(vocab, bert, tokenizer, batch_size=256):
    words = list(vocab.keys())
    vecs  = []
    bert.eval()
    for s in range(0, len(words), batch_size):
        enc = tokenizer(words[s:s+batch_size], padding=True,
                        truncation=True, max_length=16,
                        return_tensors="pt").to(DEVICE)
        vecs.append(bert(**enc).last_hidden_state[:, 1, :].cpu())
    return torch.cat(vecs, dim=0).to(DEVICE)


# ══════════════════════════════════════════════════════════════════
# 2.5  Model  (paper §3.3–3.4)
# ══════════════════════════════════════════════════════════════════

class GCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim, dropout=0.5):
        super().__init__()
        self.W  = nn.Linear(in_dim, out_dim, bias=False)
        self.dp = nn.Dropout(dropout)
    def forward(self, H, A):
        return F.relu(self.W(A @ self.dp(H)))


class GraphEncoder(nn.Module):
    def __init__(self, in_dim, out_dim, dropout=0.5, depth=1):
        super().__init__()
        dims = [in_dim] + [out_dim] * depth
        self.layers = nn.ModuleList(
            [GCNLayer(dims[i], dims[i+1], dropout) for i in range(depth)])
    def forward(self, H, A):
        out = H
        for layer in self.layers:
            new = layer(out, A)
            out = new + out if new.shape == out.shape else new
        return out.mean(dim=0)


class BERTG3CN(nn.Module):
    CO_DIM = 16; HET_DIM = 128; SYN_DIM = 128

    def __init__(self, bert_name="bert-base-uncased",
                 num_classes=2, dropout=0.5, gcn_depth=1):
        super().__init__()
        self.bert    = BertModel.from_pretrained(bert_name,
                           ignore_mismatched_sizes=True)
        H            = self.bert.config.hidden_size
        self.enc_co  = GraphEncoder(H, self.CO_DIM,  dropout, gcn_depth)
        self.enc_het = GraphEncoder(H, self.HET_DIM, dropout, gcn_depth)
        self.enc_syn = GraphEncoder(H, self.SYN_DIM, dropout, gcn_depth)
        self.fusion  = nn.Sequential(
            nn.Linear(H + self.CO_DIM + self.HET_DIM + self.SYN_DIM, H),
            nn.ReLU(),
            nn.Dropout(dropout))
        self.head = nn.Linear(H, num_classes)

    def forward(self, input_ids, attention_mask, A_co, A_het, A_syn, H_vocab):
        cls   = self.bert(input_ids=input_ids,
                          attention_mask=attention_mask
                          ).last_hidden_state[:, 0, :]
        B     = cls.size(0)
        g_co  = self.enc_co (H_vocab, A_co ).unsqueeze(0).expand(B, -1)
        g_het = self.enc_het(H_vocab, A_het).unsqueeze(0).expand(B, -1)
        g_syn = self.enc_syn(H_vocab, A_syn).unsqueeze(0).expand(B, -1)
        return self.head(
            self.fusion(torch.cat([cls, g_co, g_het, g_syn], dim=-1)))


# ══════════════════════════════════════════════════════════════════
# 2.6  Dataset & loaders
# ══════════════════════════════════════════════════════════════════

class SMSDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts   = [str(t) for t in texts]
        self.labels  = labels
        self.tok     = tokenizer
        self.max_len = max_len
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tok(self.texts[idx], max_length=self.max_len,
                       padding="max_length", truncation=True,
                       return_tensors="pt")
        return {"input_ids":      enc["input_ids"].squeeze(0),
                "attention_mask": enc["attention_mask"].squeeze(0),
                "label":          torch.tensor(self.labels[idx], dtype=torch.long)}


def make_loaders(tr_txt, tr_lbl, va_txt, va_lbl, te_txt, te_lbl,
                 tokenizer, max_len=128, batch_size=16):
    def mk(t, l, shuffle):
        return DataLoader(
            SMSDataset(t, l, tokenizer, max_len),
            batch_size=batch_size if shuffle else batch_size * 2,
            shuffle=shuffle, num_workers=2,
            pin_memory=(DEVICE.type == "cuda"))
    return mk(tr_txt, tr_lbl, True), mk(va_txt, va_lbl, False), mk(te_txt, te_lbl, False)


# ══════════════════════════════════════════════════════════════════
# 2.7  Train / Evaluate loops
# ══════════════════════════════════════════════════════════════════

def train_epoch(model, loader, optimizer, scheduler, criterion,
                A_co, A_het, A_syn, H_vocab):
    model.train()
    total_loss, preds, labels = 0.0, [], []
    for b in tqdm(loader, desc="  train", leave=False):
        ids  = b["input_ids"]     .to(DEVICE)
        mask = b["attention_mask"].to(DEVICE)
        lbl  = b["label"]        .to(DEVICE)
        optimizer.zero_grad()
        out  = model(ids, mask, A_co, A_het, A_syn, H_vocab)
        loss = criterion(out, lbl)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step(); scheduler.step()
        total_loss += loss.item()
        preds .extend(out.argmax(-1).cpu().tolist())
        labels.extend(lbl.cpu().tolist())
    return total_loss / len(loader), accuracy_score(labels, preds)


@torch.no_grad()
def eval_epoch(model, loader, criterion, A_co, A_het, A_syn, H_vocab):
    model.eval()
    total_loss, preds, probs, labels = 0.0, [], [], []
    for b in tqdm(loader, desc="  eval ", leave=False):
        ids  = b["input_ids"]     .to(DEVICE)
        mask = b["attention_mask"].to(DEVICE)
        lbl  = b["label"]        .to(DEVICE)
        out  = model(ids, mask, A_co, A_het, A_syn, H_vocab)
        total_loss += criterion(out, lbl).item()
        probs .extend(F.softmax(out, -1)[:, 1].cpu().tolist())
        preds .extend(out.argmax(-1).cpu().tolist())
        labels.extend(lbl.cpu().tolist())
    return total_loss / len(loader), accuracy_score(labels, preds), preds, probs, labels


# ══════════════════════════════════════════════════════════════════
# 2.8  Full training pipeline
# ══════════════════════════════════════════════════════════════════

def train_model(name, tr_loader, va_loader, te_loader,
                A_co, A_het, A_syn, H_vocab,
                bert_name="bert-base-uncased",
                epochs=10, lr=1e-5, weight_decay=1e-4,
                dropout=0.5, gcn_depth=1, **_):

    model     = BERTG3CN(bert_name, dropout=dropout, gcn_depth=gcn_depth).to(DEVICE)
    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    steps     = len(tr_loader) * epochs
    scheduler = get_linear_schedule_with_warmup(optimizer, int(0.1*steps), steps)
    criterion = nn.CrossEntropyLoss()

    history          = {"tr_loss":[], "va_loss":[], "tr_acc":[], "va_acc":[]}
    best_va, best_st = 0.0, None

    print(f"\n  {'Epoch':>5}  {'Tr-Loss':>9}  {'Tr-Acc':>8}  "
          f"{'Va-Loss':>9}  {'Va-Acc':>8}  {'Time':>6}")
    print(f"  {'─'*54}")

    for ep in range(1, epochs + 1):
        t0 = time.time()
        tl, ta = train_epoch(model, tr_loader, optimizer, scheduler,
                             criterion, A_co, A_het, A_syn, H_vocab)
        vl, va, _, _, _ = eval_epoch(model, va_loader, criterion,
                                      A_co, A_het, A_syn, H_vocab)
        history["tr_loss"].append(tl); history["va_loss"].append(vl)
        history["tr_acc"] .append(ta); history["va_acc"] .append(va)

        flag = "  ◀" if va > best_va else ""
        print(f"  {ep:>5}  {tl:>9.4f}  {ta:>8.4f}  "
              f"{vl:>9.4f}  {va:>8.4f}  {time.time()-t0:>5.1f}s{flag}")

        if va > best_va:
            best_va = va
            best_st = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict({k: v.to(DEVICE) for k, v in best_st.items()})
    _, te_acc, preds, probs, true = eval_epoch(
        model, te_loader, criterion, A_co, A_het, A_syn, H_vocab)

    p, r, f, _ = precision_recall_fscore_support(
        true, preds, average=None, labels=[0, 1])
    auc = roc_auc_score(true, probs)

    return dict(name=name, model=model, history=history,
                te_acc=te_acc, te_auc=auc,
                preds=preds, probs=probs, true=true,
                precision=p, recall=r, f1=f)


# ══════════════════════════════════════════════════════════════════
# 2.9  Grid-search trial helper
# ══════════════════════════════════════════════════════════════════

def gs_trial(tr_txt, tr_lbl, va_txt, va_lbl, te_txt, te_lbl,
             tokenizer, A_co, A_het, A_syn, H_vocab,
             epochs, bert_name, batch_size, max_len,
             weight_decay, lr, dropout, gcn_depth, **_):

    random.seed(SEED); np.random.seed(SEED)
    torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

    model = BERTG3CN(bert_name, dropout=dropout,
                     gcn_depth=gcn_depth).to(DEVICE)
    tr_ld, va_ld, te_ld = make_loaders(
        tr_txt, tr_lbl, va_txt, va_lbl, te_txt, te_lbl,
        tokenizer, max_len, batch_size)

    opt   = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    steps = len(tr_ld) * epochs
    sched = get_linear_schedule_with_warmup(opt, int(0.1*steps), steps)
    crit  = nn.CrossEntropyLoss()

    best_va, best_st = 0.0, None
    for _ in range(epochs):
        train_epoch(model, tr_ld, opt, sched, crit,
                    A_co, A_het, A_syn, H_vocab)
        _, va, _, _, _ = eval_epoch(model, va_ld, crit,
                                     A_co, A_het, A_syn, H_vocab)
        if va > best_va:
            best_va = va
            best_st = {k: v.cpu().clone()
                       for k, v in model.state_dict().items()}

    model.load_state_dict({k: v.to(DEVICE) for k, v in best_st.items()})
    _, acc, _, probs, true = eval_epoch(model, te_ld, crit,
                                         A_co, A_het, A_syn, H_vocab)
    auc = roc_auc_score(true, probs)
    del model; gc.collect(); torch.cuda.empty_cache()
    return round(acc * 100, 2), round(auc, 4)


# ══════════════════════════════════════════════════════════════════
# 2.10  Plot helpers
# ══════════════════════════════════════════════════════════════════

def plot_training(result):
    h  = result["history"]
    ep = range(1, len(h["tr_loss"]) + 1)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
    ax1.plot(ep, h["tr_loss"], "b-o", ms=4, label="Train")
    ax1.plot(ep, h["va_loss"], "r--o", ms=4, label="Validation")
    ax1.set(title=f"{result['name']} — Loss", xlabel="Epoch", ylabel="Loss")
    ax1.legend(); ax1.grid(alpha=.3)
    ax2.plot(ep, h["tr_acc"], "b-o", ms=4, label="Train")
    ax2.plot(ep, h["va_acc"], "r--o", ms=4, label="Validation")
    ax2.set(title=f"{result['name']} — Accuracy",
            xlabel="Epoch", ylabel="Accuracy", ylim=(0.5, 1.02))
    ax2.legend(); ax2.grid(alpha=.3)
    plt.tight_layout(); plt.show()


def plot_roc(result):
    fpr, tpr, _ = roc_curve(result["true"], result["probs"])
    plt.figure(figsize=(6, 5))
    plt.plot(fpr, tpr, lw=2,
             label=f"BERT-G3CN  AUC = {result['te_auc']:.4f}")
    plt.plot([0, 1], [0, 1], "k--", lw=1)
    plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
    plt.title(f"ROC Curve — {result['name']}")
    plt.legend(); plt.grid(alpha=.3)
    plt.tight_layout(); plt.show()


def plot_confusion(result):
    cm = confusion_matrix(result["true"], result["preds"])
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["Ham", "Spam"],
                yticklabels=["Ham", "Spam"])
    plt.title(f"Confusion Matrix — {result['name']}")
    plt.xlabel("Predicted"); plt.ylabel("Actual")
    plt.tight_layout(); plt.show()


def print_metrics(result, paper_acc=None, paper_auc=None):
    p, r, f = result["precision"], result["recall"], result["f1"]
    print(f"\n  Results — {result['name']}")
    print(f"  {'─'*52}")
    print(f"  {'Class':<8} {'Precision':>10} {'Recall':>10} {'F1-Score':>10}")
    print(f"  {'Ham'  :<8} {p[0]:>10.4f} {r[0]:>10.4f} {f[0]:>10.4f}")
    print(f"  {'Spam' :<8} {p[1]:>10.4f} {r[1]:>10.4f} {f[1]:>10.4f}")
    print(f"  {'─'*52}")
    acc_line = f"  Accuracy : {result['te_acc']*100:.2f}%"
    auc_line = f"  AUC      : {result['te_auc']:.4f}"
    if paper_acc: acc_line += f"   (paper: {paper_acc}%)"
    if paper_auc: auc_line += f"   (paper: {paper_auc})"
    print(acc_line); print(auc_line)


print("All definitions ready.")

All definitions ready.


In [ ]:
PREFIX   = os.getenv("my_prefix") 

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  Cell 10 (or top of notebook) — Imports                          ║
# ╚══════════════════════════════════════════════════════════════════╝

import json
import torch
import torch.nn.functional as F    
from transformers import BertTokenizer

# Also make sure DEVICE is defined
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# And make sure these custom functions/classes are defined or imported:
# - BERTG3CN (your model class)    
# - build_vocab_embeddings (your helper function)

# PREFIX = "uci_model"  # or "exais_model" 

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  Cell 11 — Load Model & Run Predictions                         ║
# ║  Can run in a completely fresh session.                         ║
# ║  Set PREFIX to "uci_model" or "exais_model".                   ║
# ╚══════════════════════════════════════════════════════════════════╝

# ── Configuration ─────────────────────────────────────────────────
    # change to "exais_model" for the other model
MAX_LEN  = 128  

# ── Load artifacts ────────────────────────────────────────────────
print(f"\nLoading {PREFIX} ...")

with open(f"{PREFIX}_config.json") as fh:
    _cfg = json.load(fh)

with open(f"{PREFIX}_vocab.json") as fh:
    _vocab = json.load(fh)  

_graphs   = torch.load(f"{PREFIX}_graphs.pt", map_location=DEVICE)
_A_co     = _graphs["A_co"] .to(DEVICE)
_A_het    = _graphs["A_het"].to(DEVICE)    
_A_syn    = _graphs["A_syn"].to(DEVICE)

_tokenizer = BertTokenizer.from_pretrained(_cfg["bert_name"]) 

_model = BERTG3CN(
    bert_name  = _cfg["bert_name"],
    dropout    = _cfg["dropout"],
    gcn_depth  = _cfg["gcn_depth"],
).to(DEVICE)
_model.load_state_dict(
    torch.load(f"{PREFIX}_weights.pt", map_location=DEVICE))
_model.eval()

# Build vocabulary embedding matrix for inference
print("Preparing vocabulary embeddings ...")
_H_vocab = build_vocab_embeddings(_vocab, _model.bert, _tokenizer)

print(f"  Ready.  Model trained on "
      f"{'UCI SMS' if 'uci' in PREFIX else 'ExAIS SMS'}, "
      f"vocab size = {len(_vocab):,}")


# ── Prediction function ───────────────────────────────────────────

@torch.no_grad()
def predict(messages):
    """
    Pass a single string or a list of strings.
    Returns a list of dicts: message, label, spam_prob, ham_prob, confidence.
    """
    if isinstance(messages, str):
        messages = [messages]

    enc = _tokenizer(messages, max_length=MAX_LEN, padding="max_length",
                     truncation=True, return_tensors="pt")
    ids  = enc["input_ids"]     .to(DEVICE)
    mask = enc["attention_mask"].to(DEVICE)
    probs = F.softmax(
        _model(ids, mask, _A_co, _A_het, _A_syn, _H_vocab), dim=-1
    ).cpu().numpy()

    out = []
    for msg, p in zip(messages, probs):
        out.append({
            "message"   : msg,
            "label"     : "SPAM" if p[1] > p[0] else "HAM",
            "spam_prob" : round(float(p[1]) * 100, 2),
            "ham_prob"  : round(float(p[0]) * 100, 2),
            "confidence": round(float(max(p)) * 100, 2),
        })
    return out


def show_predictions(results):
    print(f"\n  {'#':<3} {'Label':<5}  {'Conf':>6}  {'Spam%':>6}  "
          f"{'Ham%':>6}  Message")
    print(f"  {'─'*80}")
    for i, r in enumerate(results, 1):
        tag = "SPAM" if r["label"] == "SPAM" else "HAM "
        msg = r["message"][:60] + "..." if len(r["message"]) > 60 else r["message"]
        print(f"  {i:<3} {tag}  {r['confidence']:>5.1f}%  "
              f"{r['spam_prob']:>5.1f}%  {r['ham_prob']:>5.1f}%  {msg}")


# ── Demo predictions ──────────────────────────────────────────────
DEMO_MESSAGES = [
    # Spam examples
    "WINNER!! You have been selected for a £1000 cash prize. Call now!",
    "FREE entry: Win a brand new iPhone. Text WIN to 80800. T&Cs apply.",
    "Urgent: Your account has been suspended. Verify at secure-login.net",  
    "Congratulations! You've won a free holiday. Click here to claim.",
    "Your mobile number has won £500. Reply YES to claim your reward.",
    # Ham examples
    "Hey, are you coming to the meeting at 3pm?",
    "I'll be home by 7, can you start dinner?",
    "Happy birthday! Hope you have a wonderful day.",
    "The report is ready, I've sent it over to you.",  
    "Can we reschedule our call to tomorrow morning?",
    # Borderline
    "Your parcel could not be delivered. Click here to reschedule.",
    "Reminder: Your appointment is confirmed for Monday at 10am.",
    "We have a special offer just for you this weekend.",
]

print("\nRunning predictions on example messages ...")
show_predictions(predict(DEMO_MESSAGES))


# ── Interactive mode ──────────────────────────────────────────────
print("\n" + "─" * 62) 
print("  Interactive prediction  —  type any SMS message.") 
print("  Enter 'quit' to exit.")
print("─" * 62)

while True:
    user_msg = input("\n  SMS: ").strip()
    if user_msg.lower() in ("quit", "exit", "q", ""): 
        print("  Exiting.")
        break
    r = predict(user_msg)[0]
    print(f"\n  Prediction  : {r['label']}")
    print(f"  Spam        : {r['spam_prob']}%")
    print(f"  Ham         : {r['ham_prob']}%")
    print(f"  Confidence  : {r['confidence']}%")


Loading /kaggle/input/models/atul0110/m1/other/default/1/exais_model ...


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

...


  Prediction  : HAM
  Spam        : 0.13%
  Ham         : 99.87%
  Confidence  : 99.87%



  SMS:  clikc me to get money 



  Prediction  : HAM
  Spam        : 0.1%
  Ham         : 99.9%
  Confidence  : 99.9%



  SMS:  q


  Exiting.
